In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import drive
import os, pathlib

---
# Dataset 1 Pre-Processing: UN Comtrade International Trade Data

## Overview

The raw Comtrade data is split across **4 CSV files** by time period, all in the same Drive folder:

| File | Rows | Period |
|---|---|---|
| `TradeData_3_4_1988-1989.csv` | 773 | 1988–1989 |
| `TradeData_3_4_1990-2001.csv` | 26,403 | 1990–2001 |
| `TradeData_3_4_2002-2013.csv` | 42,764 | 2002–2013 |
| `TradeData_3_4_2014-2025.csv` | 38,768 | 2014–2025 |

**Combined: 108,708 rows, 47 columns.**

Each row records a country's total imports or exports of a specific HS commodity chapter in a given year, reported to the World. The key issues to resolve before loading into PostgreSQL:

| Issue | Description | Fix |
|---|---|---|
| Multiple HS revisions | Same chapter appears under H1–H6 for overlapping years | Keep most recent revision per (country, year, chapter, flow) |
| `TOTAL` rows | Some rows aggregate all commodity chapters | Drop them (redundant) |
| Sparse/missing `primaryValue` | A small fraction have no trade value | Drop them |
| Pre-1995 data | Thin coverage before 1995 | Filter to year ≥ 1995 |
| 39 unused columns | Weight, quantity, flags, descriptions not needed for analysis | Drop them |

The goal is a clean fact table with schema:
```
trade(country_code, year, trade_flow, hs_chapter, hs_revision, trade_value_usd, hs_chapter_desc)
```


## Step 1 — Load All Four Files and Concatenate

We use `glob` to automatically find every file matching `TradeData*.csv` in the folder, so the code still works if more files are added later.

The files use `latin-1` encoding (not UTF-8) because some country and commodity descriptions contain special characters from older Windows code pages.

We again load as `dtype=str` to prevent premature type coercion.


In [ ]:
import pandas as pd
import os
import glob

drive.mount('/content/drive')

TRADE_FOLDER = '/content/drive/MyDrive/CIS5500 Final Project Data/trade data'

csv_files = sorted(glob.glob(os.path.join(TRADE_FOLDER, 'TradeData*.csv')))
print(f"Found {len(csv_files)} file(s):")

chunks = []
for f in csv_files:
    chunk = pd.read_csv(f, encoding='latin-1', dtype=str, index_col=False)
    print(f"  {os.path.basename(f)}: {len(chunk):,} rows")
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"\nCombined raw shape: {df.shape}")
df.head(2)

Mounted at /content/drive
Found 4 file(s):
  TradeData_3_4_1988-1989.csv: 773 rows
  TradeData_3_4_1990-2001.csv: 26,403 rows
  TradeData_3_4_2002-2013.csv: 42,764 rows
  TradeData_3_4_2014-2025.csv: 38,768 rows

Combined raw shape: (108708, 47)


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,19880101,1988,52,1988,36,AUS,Australia,M,...,NaN,false,NaN,false,NaN,148112063,148112063,0,false,false
1,C,A,19880101,1988,52,1988,36,AUS,Australia,X,...,NaN,false,NaN,false,NaN,109552292,109552292,0,false,false


## Step 2 — Drop TOTAL Rows

The `aggrLevel` column indicates the level of aggregation in the HS hierarchy:
- `aggrLevel = 0` → **TOTAL** — sums all commodity chapters into one row per country-year-flow
- `aggrLevel = 2` → **HS 2-digit chapter** — the level we want (e.g., chapter 85 = Electrical machinery)

We drop `aggrLevel = 0` rows because they are redundant roll-ups. Keeping them alongside chapter-level rows would cause double-counting in any aggregation query.

After examining the data, we confirmed that only values `'0'` and `'2'` appear in `aggrLevel` — there are no intermediate levels in this dataset.


In [ ]:
print(f"aggrLevel value counts before filtering:")
print(df['aggrLevel'].value_counts())

# Keep only HS 2-digit chapter rows; discard TOTAL
df = df[df['aggrLevel'].isin(['2'])]

print(f"\nAfter dropping TOTAL rows (aggrLevel=0): {df.shape}")
print(f"Unique HS chapters remaining: {df['cmdCode'].nunique()}")
print(f"HS chapters: {sorted(df['cmdCode'].unique())}")

aggrLevel value counts before filtering:
aggrLevel
2    98195
0    10513
Name: count, dtype: int64

After dropping TOTAL rows (aggrLevel=0): (98195, 47)
Unique HS chapters remaining: 10
HS chapters: ['16', '19', '20', '22', '24', '30', '50', '85', '87', '93']


## Step 3 — Handle Multiple HS Revisions

The Harmonized System (HS) is periodically updated. Comtrade records which HS revision was used for each observation in the `classificationCode` column (H0 through H6, where H6 is the most recent — 2022 edition).

Because the data spans 1988–2025, earlier years use older HS revisions:
- 1988–1995: H0 or H1
- 1996–2001: H1 or H2
- 2002–2006: H2 or H3
- 2007–2011: H3 or H4
- 2012–2016: H4 or H5
- 2017–2021: H5 or H6
- 2022+: H6

For our analysis, chapter numbers are **stable enough** across revisions at the 2-digit level (e.g., chapter 85 is always Electrical machinery across all HS versions). However, a small number of country-year-chapter rows appear in multiple revisions. We resolve this by keeping only the **most recent revision** for each (country, year, chapter, flow) combination.


In [ ]:
print("HS revision distribution before deduplication:")
print(df['classificationCode'].value_counts().sort_index())

# Assign a numeric rank: H6 > H5 > H4 > H3 > H2 > H1 > H0
revision_order = {'H6': 6, 'H5': 5, 'H4': 4, 'H3': 3, 'H2': 2, 'H1': 1, 'H0': 0}
df['revision_rank'] = df['classificationCode'].map(revision_order).fillna(0).astype(int)

# Sort so highest revision comes first, then deduplicate keeping first
df = df.sort_values('revision_rank', ascending=False)
df = df.drop_duplicates(
    subset=['reporterISO', 'refYear', 'cmdCode', 'flowCode'],
    keep='first'
)

print(f"\nAfter deduplication (prefer most recent HS revision): {df.shape}")

HS revision distribution before deduplication:
classificationCode
H0    13130
H1    16590
H2    15981
H3    15403
H4    15612
H5    14220
H6     7259
Name: count, dtype: int64

After deduplication (prefer most recent HS revision): (98195, 48)


## Step 4 — Cast Types and Drop Missing Trade Values

We convert `refYear` to integer and `primaryValue` to float.

**Why `primaryValue` and not `cifvalue` or `fobvalue`?**
- Imports are valued **CIF** (cost + insurance + freight) — so `cifvalue` is populated for import rows
- Exports are valued **FOB** (free on board) — so `fobvalue` is populated for export rows
- `primaryValue` is whichever of the two applies for each row — it is **always populated**

After casting, we drop the small fraction of rows where `primaryValue` has no numeric value.


In [ ]:
df['refYear']      = df['refYear'].astype(int)
df['primaryValue'] = pd.to_numeric(df['primaryValue'], errors='coerce')

missing_val = df['primaryValue'].isna().sum()
print(f"Rows with no primaryValue: {missing_val:,}")

df = df[df['primaryValue'].notna()]
print(f"After dropping rows with no trade value: {df.shape}")

Rows with no primaryValue: 0
After dropping rows with no trade value: (98195, 48)


## Step 5 — Filter to Year ≥ 1995

We apply the same year filter as ILOSTAT to ensure the two datasets share a consistent temporal window. Pre-1995 trade data has only 773 rows (1988–1989 file) and sparse 1990–1994 coverage — too thin to be useful for cross-dataset joins.


In [ ]:
print(f"Row count by year range before filter:")
for decade in [1988, 1990, 1995, 2000, 2010, 2020]:
    end = min(decade + (5 if decade == 1988 else 10), 2026)
    n = df[(df['refYear'] >= decade) & (df['refYear'] < end)].shape[0]
    print(f"  {decade}–{end-1}: {n:,}")

df = df[df['refYear'] >= 1995]
print(f"\nAfter filtering to year >= 1995: {df.shape}")

Row count by year range before filter:
  1988–1992: 3,084
  1990–1999: 17,675
  1995–2004: 27,786
  2000–2009: 31,867
  2010–2019: 32,592
  2020–2025: 15,361

After filtering to year >= 1995: (91955, 48)


## Step 6a — Derive HS Section and Extract Lookup Tables

The Comtrade data does not include HS section numbers directly. We derive
`hs_section` (integer 1–21) and `section_label` from the 2-digit `cmdCode` using the
standard HS section groupings, then extract the **Country**, **HS_Section**, and
**HS_Commodity** lookup tables **before** Step 6c narrows `df` to only the fact
columns (which would otherwise lose `cmdDesc`, `reporterDesc`, etc.).


In [ ]:
# Standard HS section groupings (chapters → section number + label)
hs_section_data = [
    (range(1,  6),  1,  'Live Animals and Animal Products'),
    (range(6,  15), 2,  'Vegetable Products'),
    ([15],          3,  'Animal or Vegetable Fats and Oils'),
    (range(16, 25), 4,  'Prepared Foodstuffs; Beverages and Tobacco'),
    (range(25, 28), 5,  'Mineral Products'),
    (range(28, 39), 6,  'Products of the Chemical Industries'),
    (range(39, 41), 7,  'Plastics and Rubber'),
    (range(41, 44), 8,  'Raw Hides, Skins, Leather and Furskins'),
    (range(44, 47), 9,  'Wood and Articles of Wood'),
    (range(47, 50), 10, 'Pulp of Wood; Paper and Paperboard'),
    (range(50, 64), 11, 'Textiles and Textile Articles'),
    (range(64, 68), 12, 'Footwear, Headgear, Umbrellas'),
    (range(68, 71), 13, 'Articles of Stone, Plaster, Cement, Glass'),
    ([71],          14, 'Natural Pearls, Precious Stones and Metals'),
    (range(72, 84), 15, 'Base Metals and Articles Thereof'),
    (range(84, 86), 16, 'Machinery and Electrical Equipment'),
    (range(86, 90), 17, 'Vehicles, Aircraft, Vessels'),
    (range(90, 93), 18, 'Optical, Photographic, Medical Instruments'),
    ([93],          19, 'Arms and Ammunition'),
    (range(94, 97), 20, 'Miscellaneous Manufactured Articles'),
    ([97],          21, 'Works of Art and Antiques'),
]

chapter_to_section = {}
chapter_to_label   = {}
for chapters, sec_num, label in hs_section_data:
    for ch in chapters:
        key = str(ch).zfill(2)  # e.g. 1 -> '01', 85 -> '85'
        chapter_to_section[key] = sec_num
        chapter_to_label[key]   = label

# Apply mapping — cmdCode is the 2-digit HS chapter (not yet renamed)
df['hs_section']    = df['cmdCode'].map(chapter_to_section)
df['section_label'] = df['cmdCode'].map(chapter_to_label)

unmapped = df['hs_section'].isna().sum()
print(f'Unmapped chapters (hs_section will be NULL): {unmapped}')
print(f'Unique sections found: {sorted(df["hs_section"].dropna().unique())}')

# ── HS_Section lookup (fixes 3NF: section_label depends on section, not on hs_code)
hs_section_df = (
    df[['hs_section', 'section_label']]
    .drop_duplicates()
    .dropna()
    .sort_values('hs_section')
    .reset_index(drop=True)
)
print(f'\nHS_Section rows: {len(hs_section_df)}')
print(hs_section_df.to_string(index=False))

# ── HS_Commodity lookup (references hs_section as FK; no section_label here)
hs_commodity_df = (
    df[['cmdCode', 'cmdDesc', 'hs_section']]
    .rename(columns={'cmdCode': 'hs_code', 'cmdDesc': 'hs_description'})
    .drop_duplicates(subset=['hs_code'])
    .dropna(subset=['hs_code'])
    .sort_values('hs_code')
    .reset_index(drop=True)
)
print(f'\nHS_Commodity rows: {len(hs_commodity_df)}')

# ── Country lookup (grab reporterDesc before it is dropped in Step 6c)
country_df = (
    df[['reporterISO', 'reporterDesc']]
    .rename(columns={'reporterISO': 'country_code', 'reporterDesc': 'country_name'})
    .drop_duplicates(subset=['country_code'])
    .dropna(subset=['country_code'])
    .sort_values('country_code')
    .reset_index(drop=True)
)
country_df['region'] = None  # region can be joined from an ISO 3166 reference later
print(f'\nCountry rows: {len(country_df)}')


Unmapped chapters (hs_section will be NULL): 0
Unique sections found: [np.int64(4), np.int64(6), np.int64(11), np.int64(16), np.int64(17), np.int64(19)]

HS_Section rows: 6
 hs_section                              section_label
          4 Prepared Foodstuffs; Beverages and Tobacco
          6        Products of the Chemical Industries
         11              Textiles and Textile Articles
         16         Machinery and Electrical Equipment
         17                Vehicles, Aircraft, Vessels
         19                        Arms and Ammunition

HS_Commodity rows: 10

Country rows: 209


## Step 6b — Save Lookup Tables to CSV

Persist the three lookup tables to Drive before the fact table columns are narrowed
in Step 6c.


In [ ]:
LOOKUP_FOLDER = '/content/drive/MyDrive/CIS5500 Final Project Data/lookup tables'
os.makedirs(LOOKUP_FOLDER, exist_ok=True)

hs_section_df.to_csv(  f'{LOOKUP_FOLDER}/hs_section.csv',   index=False)
hs_commodity_df.to_csv(f'{LOOKUP_FOLDER}/hs_commodity.csv', index=False)
country_df.to_csv(     f'{LOOKUP_FOLDER}/country.csv',      index=False)

print('Lookup tables saved:')
print(f'  hs_section.csv   — {len(hs_section_df)} rows')
print(f'  hs_commodity.csv — {len(hs_commodity_df)} rows')
print(f'  country.csv      — {len(country_df)} rows')


Lookup tables saved:
  hs_section.csv   — 6 rows
  hs_commodity.csv — 10 rows
  country.csv      — 209 rows


## Step 6c — Select and Rename Final Columns

We drop 39 of the original 47 columns. The ones we keep are:

| Original column | Renamed to | Why |
|---|---|---|
| `reporterISO` | `country_code` | ISO 3-letter code — reporting country, join key to ILOSTAT |
| `partnerISO` | `partner_code` | ISO 3-letter code — trading partner, required for Top Trading Partners page |
| `refYear` | `year` | Join key |
| `flowCode` | `trade_flow` | 'M' = Import, 'X' = Export |
| `flowDesc` | `trade_flow_desc` | Human-readable label |
| `cmdCode` | `hs_chapter` | 2-digit HS chapter — join key via HS-to-ISIC crosswalk |
| `cmdDesc` | `hs_chapter_desc` | Human-readable commodity description |
| `classificationCode` | `hs_revision` | H0–H6, useful for audit trail |
| `primaryValue` | `trade_value_usd` | The trade value in USD |

The final PostgreSQL table will be:
```sql
CREATE TABLE Trade_Flow (
    country_code     CHAR(3)       NOT NULL REFERENCES Country(country_code),
    partner_code     CHAR(3)       NOT NULL REFERENCES Country(country_code),
    year             SMALLINT      NOT NULL,
    trade_flow       CHAR(1)       NOT NULL CHECK (trade_flow IN ('X', 'M')),
    trade_flow_desc  VARCHAR(10),
    hs_chapter       VARCHAR(5)    NOT NULL REFERENCES HS_Commodity(hs_code),
    hs_chapter_desc  TEXT,
    hs_revision      VARCHAR(3),
    trade_value_usd  NUMERIC(20,2),
    PRIMARY KEY (country_code, partner_code, year, trade_flow, hs_chapter)
);
```


In [ ]:
df = df.rename(columns={
    'reporterISO':        'country_code',
    'partnerISO':         'partner_code',
    'refYear':            'year',
    'flowCode':           'trade_flow',
    'flowDesc':           'trade_flow_desc',
    'cmdCode':            'hs_chapter',
    'cmdDesc':            'hs_chapter_desc',
    'classificationCode': 'hs_revision',
    'primaryValue':       'trade_value_usd',
})

df = df[[
    'country_code',
    'partner_code',
    'year',
    'trade_flow',
    'trade_flow_desc',
    'hs_chapter',
    'hs_chapter_desc',
    'hs_revision',
    'trade_value_usd',
]]

print(f'Final schema:')
print(df.dtypes)
print(f'\nSample partner_code values: {df["partner_code"].value_counts().head(5).to_dict()}')


Final schema:
country_code        object
partner_code        object
year                 int64
trade_flow          object
trade_flow_desc     object
hs_chapter          object
hs_chapter_desc     object
hs_revision         object
trade_value_usd    float64
dtype: object

Sample partner_code values: {'W00': 91955}


## Summary and Descriptive Statistics


In [ ]:
print(f"{'='*50}")
print(f"Trade Cleaned Dataset Summary")
print(f"{'='*50}")
print(f"Total rows:      {len(df):,}")
print(f"Countries:       {df['country_code'].nunique()}")
print(f"Years:           {df['year'].min()} – {df['year'].max()}")
print(f"HS chapters:     {sorted(df['hs_chapter'].unique())}")
print(f"Trade flows:     {df['trade_flow'].value_counts().to_dict()}")
print(f"HS revisions:    {df['hs_revision'].value_counts().sort_index().to_dict()}")
print()
print("Descriptive statistics for trade_value_usd:")
print(df['trade_value_usd'].describe().apply(lambda x: f'{x:,.2f}'))
print()
print("Sample rows:")
display(df.sample(5, random_state=42))

Trade Cleaned Dataset Summary
Total rows:      91,955
Countries:       209
Years:           1995 – 2025
HS chapters:     ['16', '19', '20', '22', '24', '30', '50', '85', '87', '93']
Trade flows:     {'M': 48589, 'X': 43366}
HS revisions:    {'H0': 6890, 'H1': 16590, 'H2': 15981, 'H3': 15403, 'H4': 15612, 'H5': 14220, 'H6': 7259}

Descriptive statistics for trade_value_usd:
count             91,955.00
mean       2,420,513,617.75
std       17,638,233,402.18
min                    0.10
25%            2,455,023.50
50%           32,082,291.00
75%          270,078,591.35
max      954,783,811,680.00
Name: trade_value_usd, dtype: object

Sample rows:


,country_code,partner_code,year,trade_flow,trade_flow_desc,hs_chapter,hs_chapter_desc,hs_revision,trade_value_usd
53668,MDG,W00,2009,X,Export,20,"Preparations of vegetables, fruit, nuts or oth...",H3,1.057731e+07
23607,TKM,W00,2000,X,Export,50,Silk,H1,3.464759e+06
38935,GHA,W00,2005,M,Import,24,Tobacco and manufactured tobacco substitutes,H2,8.997450e+05
22684,NOR,W00,2000,X,Export,50,Silk,H1,1.178370e+05
84836,BRA,W00,2018,X,Export,50,Silk,H5,3.374345e+07


In [ ]:
df.head(10)

,country_code,partner_code,year,trade_flow,trade_flow_desc,hs_chapter,hs_chapter_desc,hs_revision,trade_value_usd
108675,PRY,W00,2025,M,Import,85,Electrical machinery and equipment and parts t...,H6,3.661714e+09
108689,ZAF,W00,2025,X,Export,16,"Meat, fish, crustaceans, molluscs or other aqu...",H6,1.296924e+08
108705,ZAF,W00,2025,X,Export,87,Vehicles; other than railway or tramway rollin...,H6,1.615918e+10
108704,ZAF,W00,2025,M,Import,87,Vehicles; other than railway or tramway rollin...,H6,9.282394e+09
108688,ZAF,W00,2025,M,Import,16,"Meat, fish, crustaceans, molluscs or other aqu...",H6,1.447878e+08
108683,PRY,W00,2025,M,Import,93,Arms and ammunition; parts and accessories the...,H6,2.715863e+06
108682,NOR,W00,2025,X,Export,93,Arms and ammunition; parts and accessories the...,H6,9.638945e+08
108703,ZAF,W00,2025,X,Export,85,Electrical machinery and equipment and parts t...,H6,2.512448e+09
108702,ZAF,W00,2025,M,Import,85,Electrical machinery and equipment and parts t...,H6,1.104314e+10
108701,ZAF,W00,2025,X,Export,50,Silk,H6,1.289338e+06


In [ ]:
# Export
OUTPUT_PATH = os.path.join(TRADE_FOLDER, 'trade_cleaned.csv')
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")

Saved to: /content/drive/MyDrive/CIS5500 Final Project Data/trade data/trade_cleaned.csv


## Step 7 — Build Commodity_Sector_Mapping Crosswalk

The `Commodity_Sector_Mapping` table links HS commodity chapters to ISIC employment
sectors. This is the join bridge between the trade and employment datasets, and is
required by the Trade Volume vs. Employment, Export Composition, and Workforce
Specialization pages.

The mapping follows the standard HS-to-ISIC concordance used by international
economic research (e.g., UN, World Bank). Some chapters map to multiple ISIC sections
(e.g., chapter 84 covers both manufacturing and services).


In [ ]:
# HS chapter (2-digit) → ISIC Rev.4 section letter
# Based on standard UN HS-to-ISIC concordance
crosswalk_rows = [
    # Section A — Agriculture, forestry and fishing
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'A', 'mapping_notes': 'Agriculture/forestry/fishing products'} for c in range(1, 6)],
    {'hs_code': '06', 'isic_section': 'A', 'mapping_notes': 'Live trees — Agriculture'},
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'A', 'mapping_notes': 'Vegetable products — Agriculture'} for c in range(7, 15)],
    {'hs_code': '15', 'isic_section': 'A', 'mapping_notes': 'Animal/vegetable fats — Agriculture'},
    # Section B — Mining and quarrying
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'B', 'mapping_notes': 'Mineral products — Mining'} for c in range(25, 28)],
    {'hs_code': '71', 'isic_section': 'B', 'mapping_notes': 'Precious stones/metals — Mining'},
    {'hs_code': '26', 'isic_section': 'B', 'mapping_notes': 'Ores, slag and ash — Mining'},
    # Section C — Manufacturing (bulk of HS chapters)
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Prepared foodstuffs — Manufacturing'} for c in range(16, 25)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Chemical products — Manufacturing'} for c in range(28, 39)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Plastics/rubber — Manufacturing'} for c in range(39, 41)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Leather/hides — Manufacturing'} for c in range(41, 44)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Wood products — Manufacturing'} for c in range(44, 47)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Paper products — Manufacturing'} for c in range(47, 50)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Textiles — Manufacturing'} for c in range(50, 64)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Footwear/headgear — Manufacturing'} for c in range(64, 68)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Stone/glass — Manufacturing'} for c in range(68, 71)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Base metals — Manufacturing'} for c in range(72, 84)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Machinery/electrical — Manufacturing'} for c in range(84, 86)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Vehicles/transport equip — Manufacturing'} for c in range(86, 90)],
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Instruments — Manufacturing'} for c in range(90, 93)],
    {'hs_code': '93', 'isic_section': 'C', 'mapping_notes': 'Arms and ammunition — Manufacturing'},
    *[{'hs_code': str(c).zfill(2), 'isic_section': 'C', 'mapping_notes': 'Misc manufactured — Manufacturing'} for c in range(94, 97)],
    {'hs_code': '97', 'isic_section': 'R', 'mapping_notes': 'Works of art — Arts/entertainment (ISIC R)'},
]

crosswalk_df = pd.DataFrame(crosswalk_rows).drop_duplicates(subset=['hs_code', 'isic_section'])

# Filter to only chapters that actually exist in trade data
crosswalk_df = crosswalk_df[crosswalk_df['hs_code'].isin(set(hs_commodity_df['hs_code']))]

print(f'Crosswalk rows: {len(crosswalk_df)}')
print(f'ISIC sections covered: {sorted(crosswalk_df["isic_section"].unique())}')
print(crosswalk_df.head(10))

# Save
crosswalk_df.to_csv(f'{LOOKUP_FOLDER}/commodity_sector_mapping.csv', index=False)
print(f'\nSaved commodity_sector_mapping.csv — {len(crosswalk_df)} rows')


Crosswalk rows: 10
ISIC sections covered: ['C']
   hs_code isic_section                             mapping_notes
20      16            C       Prepared foodstuffs — Manufacturing
23      19            C       Prepared foodstuffs — Manufacturing
24      20            C       Prepared foodstuffs — Manufacturing
26      22            C       Prepared foodstuffs — Manufacturing
28      24            C       Prepared foodstuffs — Manufacturing
31      30            C         Chemical products — Manufacturing
51      50            C                  Textiles — Manufacturing
85      85            C      Machinery/electrical — Manufacturing
87      87            C  Vehicles/transport equip — Manufacturing
93      93            C       Arms and ammunition — Manufacturing

Saved commodity_sector_mapping.csv — 10 rows


# Dataset 2 Pre-Processing: ILOSTAT Employment by Sex and Economic Activity

## Overview

The raw ILOSTAT file (`EMP_TEMP_SEX_ECO_NB_A`) contains **331,658 rows** and **11 columns**. Each row is one observation of employed persons (in thousands) for a specific country, year, industry sector, and sex category.

Before we can load this into a relational database, several issues need to be resolved:

| Issue | Description | Fix |
|---|---|---|
| Multiple classification schemes | Data uses both ISIC Rev.3 and Rev.4 for the same countries | Keep ISIC4 where available, fall back to ISIC3 |
| Roll-up / aggregate rows | `ECO_SECTOR_*` and `ECO_AGGREGATE_*` rows are pre-computed totals that cause double-counting | Drop them |
| TOTAL and unknown rows | Within ISIC, `_TOTAL` and `_X` rows are redundant or uninformative | Drop them |
| Unreliable observations | `obs_status = 'U'` means the ILO has flagged the value as unreliable | Drop them |
| Missing values | Some rows have no numeric `obs_value` | Drop them |
| Sparse early years | Data before 1995 is thin and doesn't overlap well with trade data | Filter to year ≥ 1995 |
| Unnecessary columns | `source`, `indicator`, and the three `note_*` columns add no analytical value | Drop them |

The goal is a clean, normalized fact table with schema:
```
employment(country_code, year, sex, isic_section, isic_version, employment_thousands, data_quality_flag)
```


## Step 1 — Mount Google Drive and Load Raw Data

We read the entire file as `dtype=str` first. This prevents pandas from silently coercing mixed-type columns (e.g., `obs_value` contains both numeric strings and empty strings) before we have a chance to handle them explicitly.

The file uses a UTF-8 BOM header (byte-order mark), which is why we specify `encoding='utf-8-sig'` — without it, the first column name would have a hidden `﻿` prefix that breaks column lookups.


In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/CIS5500 Final Project Data/ilostat data/EMP_TEMP_SEX_ECO_NB_A-20260327T1607.csv'

df = pd.read_csv(FILE_PATH, encoding='utf-8-sig', dtype=str)

print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw shape: (331658, 11)
Columns: ['ref_area', 'source', 'indicator', 'sex', 'classif1', 'time', 'obs_value', 'obs_status', 'note_classif', 'note_indicator', 'note_source']


,ref_area,source,indicator,sex,classif1,time,obs_value,obs_status,note_classif,note_indicator,note_source
0,ABW,BA:829,EMP_TEMP_SEX_ECO_NB,SEX_T,ECO_SECTOR_TOTAL,2011,47.915,NaN,NaN,NaN,NaN
1,ABW,BA:829,EMP_TEMP_SEX_ECO_NB,SEX_T,ECO_SECTOR_AGR,2011,0.286,NaN,NaN,NaN,NaN
2,ABW,BA:829,EMP_TEMP_SEX_ECO_NB,SEX_T,ECO_SECTOR_NAG,2011,47.629,NaN,NaN,NaN,NaN


## Step 2 — Keep Only ISIC Industry Rows

The `classif1` column contains one of four classification schemes:

| Prefix | Scheme | What it is |
|---|---|---|
| `ECO_ISIC4_*` | ISIC Rev.4 | Modern standard, 21 sections (A–U) |
| `ECO_ISIC3_*` | ISIC Rev.3 | Older standard, used by some countries pre-2008 |
| `ECO_SECTOR_*` | Broad sectors | Agriculture / Industry / Services — pre-aggregated from ISIC |
| `ECO_AGGREGATE_*` | Custom aggregates | Manufacturing, Market services, etc. — also pre-aggregated |

We drop `ECO_SECTOR_*` and `ECO_AGGREGATE_*` entirely. These are roll-up rows: their values are already sums of the ISIC rows, so including them alongside ISIC rows would cause double-counting in any `SUM()` query.

Within the ISIC rows, we also drop:
- `_TOTAL` — the within-scheme total, already a sum of the individual sections
- `_X` — "not elsewhere classified" / unknown industry, not meaningful for joins


In [ ]:
# Keep only rows where classif1 starts with ECO_ISIC3_ or ECO_ISIC4_
df = df[df['classif1'].str.startswith(('ECO_ISIC3_', 'ECO_ISIC4_'))]

# Drop TOTAL and X (unknown) sections
df = df[~df['classif1'].str.endswith(('TOTAL', '_X'))]

print(f"After keeping ISIC rows only (dropping SECTOR, AGGREGATE, TOTAL, X): {df.shape}")
print(f"Unique classif1 values remaining: {df['classif1'].nunique()}")
print(df['classif1'].unique()[:10], '...')

After keeping ISIC rows only (dropping SECTOR, AGGREGATE, TOTAL, X): (162872, 11)
Unique classif1 values remaining: 38
['ECO_ISIC3_A' 'ECO_ISIC3_D' 'ECO_ISIC3_E' 'ECO_ISIC3_F' 'ECO_ISIC3_G'
 'ECO_ISIC3_H' 'ECO_ISIC3_I' 'ECO_ISIC3_J' 'ECO_ISIC3_K' 'ECO_ISIC3_L'] ...


## Step 3 — Extract ISIC Version and Section Letter

Each `classif1` value encodes both the ISIC revision and the section letter:
- `ECO_ISIC4_C` → revision **4**, section **C** (Manufacturing)
- `ECO_ISIC3_A` → revision **3**, section **A** (Agriculture)

We extract these into two new columns:
- `isic_version` (integer: 3 or 4) — needed for the deduplication step
- `isic_section` (single letter: A–U) — this becomes the **join key** when linking to trade data via the HS-to-ISIC crosswalk


In [ ]:
# Extract revision number: ECO_ISIC4_C → 4
df['isic_version'] = df['classif1'].str.extract(r'ECO_ISIC(\d)_').astype(int)

# Extract section letter: ECO_ISIC4_C → 'C'
df['isic_section'] = df['classif1'].str.split('_').str[-1]

print("Sample of extracted values:")
df[['classif1', 'isic_version', 'isic_section']].drop_duplicates().head(10)

Sample of extracted values:


,classif1,isic_version,isic_section
43,ECO_ISIC3_A,3,A
44,ECO_ISIC3_D,3,D
45,ECO_ISIC3_E,3,E
46,ECO_ISIC3_F,3,F
47,ECO_ISIC3_G,3,G
48,ECO_ISIC3_H,3,H
49,ECO_ISIC3_I,3,I
50,ECO_ISIC3_J,3,J
51,ECO_ISIC3_K,3,K
52,ECO_ISIC3_L,3,L


## Step 4 — Deduplicate: Prefer ISIC Rev.4 Over Rev.3

Some countries reported data under both ISIC Rev.3 and Rev.4 for the same year. If we kept both, we would have two rows for the same (country, year, sex, section) combination — a logical duplicate even if the exact numbers differ slightly due to reclassification.

**Strategy:** For any (country, year, sex, section) group that has both ISIC3 and ISIC4 rows, keep only the ISIC4 row. ISIC Rev.4 is the current international standard (adopted around 2008), so it is preferred.

We implement this by sorting so ISIC4 rows come first, then calling `drop_duplicates(keep='first')` on the natural key.


In [ ]:
# Sort descending by version so ISIC4 rows appear before ISIC3 rows
df = df.sort_values('isic_version', ascending=False)

# Drop duplicates on the natural key — keep the first (= highest version)
df = df.drop_duplicates(
    subset=['ref_area', 'time', 'sex', 'isic_section'],
    keep='first'
)

print(f"After deduplication (prefer ISIC4 over ISIC3): {df.shape}")
print(f"ISIC version breakdown after dedup:")
print(df['isic_version'].value_counts().sort_index())

After deduplication (prefer ISIC4 over ISIC3): (162653, 13)
ISIC version breakdown after dedup:
isic_version
3     57371
4    105282
Name: count, dtype: int64


## Step 5 — Cast Column Types

At this point we convert from the all-string representation we loaded in Step 1:
- `time` → **integer** (year, e.g. 2010)
- `obs_value` → **float** — we use `errors='coerce'` so that empty strings and non-numeric entries become `NaN` instead of raising an exception. We'll drop those `NaN` rows in the next step.


In [ ]:
df['time']      = df['time'].astype(int)
df['obs_value'] = pd.to_numeric(df['obs_value'], errors='coerce')

print(f"dtypes after casting:")
print(df[['time', 'obs_value']].dtypes)
print(f"\nRows where obs_value is NaN (no value reported): {df['obs_value'].isna().sum():,}")

dtypes after casting:
time           int64
obs_value    float64
dtype: object

Rows where obs_value is NaN (no value reported): 4,896


## Step 6 — Drop Missing and Unreliable Observations

The `obs_status` column carries ILO quality flags:

| Value | Meaning | Action |
|---|---|---|
| *(blank)* | Normal, directly reported value | **Keep** |
| `B` | Break in series — methodology changed this year, value is still valid | **Keep** (flag preserved) |
| `U` | Unreliable — ILO has assessed this value as not fit for use | **Drop** |

We also drop any rows where `obs_value` is `NaN` (no number was reported at all). These account for ~2% of rows.


In [ ]:
# Drop rows with no numeric employment value
df = df[df['obs_value'].notna()]

# Drop rows flagged unreliable by ILO
df = df[df['obs_status'] != 'U']

print(f"After dropping missing values and unreliable observations: {df.shape}")
print(f"\nobs_status breakdown in remaining rows:")
print(df['obs_status'].value_counts(dropna=False).rename({
    '': 'normal (blank)',
    'B': 'break in series'
}))

After dropping missing values and unreliable observations: (151867, 13)

obs_status breakdown in remaining rows:
obs_status
NaN                141748
break in series     10119
Name: count, dtype: int64


## Step 7 — Filter to Year ≥ 1995

The UN Comtrade trade dataset starts from 1988, but ILOSTAT coverage is very sparse before the mid-1990s — fewer than 10,000 rows cover the entire period 1947–1989. From 1995 onward, coverage is dense and consistent for the countries we care about.

Filtering to `year >= 1995` gives us a **30-year window (1995–2025)** with strong overlap with the trade data, while keeping the cleaned row count comfortably above the 100,000-row threshold.


In [ ]:
df = df[df['time'] >= 1995]

print(f"After filtering to year >= 1995: {df.shape}")
print(f"Year range: {df['time'].min()} – {df['time'].max()}")
print(f"Rows per decade:")
for decade in [1990, 2000, 2010, 2020]:
    n = df[(df['time'] >= decade) & (df['time'] < decade + 10)].shape[0]
    print(f"  {decade}s: {n:,}")

After filtering to year >= 1995: (146731, 13)
Year range: 1995 – 2025
Rows per decade:
  1990s: 12,022
  2000s: 41,067
  2010s: 62,154
  2020s: 31,488


## Step 8 — Select and Rename Final Columns

We drop the columns that carry no analytical value for our project:
- `source` — internal ILO survey code (e.g., `BA:829`), not needed for database queries
- `indicator` — constant (`EMP_TEMP_SEX_ECO_NB`) across every row
- `classif1` — superseded by the cleaner `isic_section` and `isic_version` columns
- `note_classif`, `note_indicator`, `note_source` — mostly empty metadata notes

The final schema maps directly to the `employment` table we will create in PostgreSQL:
```sql
CREATE TABLE employment (
    country_code       CHAR(3),
    year               SMALLINT,
    sex                VARCHAR(10),
    isic_section       CHAR(1),
    isic_version       SMALLINT,
    employment_thousands NUMERIC(12,3),
    data_quality_flag  CHAR(1),
    PRIMARY KEY (country_code, year, sex, isic_section)
);
```


In [ ]:
# Fix 1 — Normalize sex codes to single characters
# Raw ILOSTAT values are SEX_M / SEX_F / SEX_T; DDL CHECK constraint uses 'M'/'F'/'T'
df['sex'] = df['sex'].map({'SEX_M': 'M', 'SEX_F': 'F', 'SEX_T': 'T'})
df = df[df['sex'].notna()]  # drop SEX_O (other/not stated) — very rare
print(f'Sex value counts after normalization: {df["sex"].value_counts().to_dict()}')

df = df.rename(columns={
    'ref_area':   'country_code',
    'time':       'year',
    'obs_value':  'employment_thousands',
    'obs_status': 'data_quality_flag',
})

df = df[[
    'country_code',          # ISO 3-letter — direct join key to trade data
    'year',
    'sex',                   # SEX_T / SEX_M / SEX_F
    'isic_section',          # A–U  — join key via HS-to-ISIC crosswalk
    'isic_version',          # 3 or 4
    'employment_thousands',  # numeric value
    'data_quality_flag',     # '' = normal, 'B' = break in series
]]

print(f"Final schema:")
print(df.dtypes)
# Fix 2 — Convert empty string obs_status to NULL
# Normal observations have obs_status = '' (empty string); store as NULL in PostgreSQL
df['data_quality_flag'] = df['data_quality_flag'].replace('', None)
print(f'\ndata_quality_flag breakdown after NULL conversion:')
print(df['data_quality_flag'].value_counts(dropna=False))


Sex value counts after normalization: {'T': 51955, 'M': 48492, 'F': 46284}
Final schema:
country_code             object
year                      int64
sex                      object
isic_section             object
isic_version              int64
employment_thousands    float64
data_quality_flag        object
dtype: object

data_quality_flag breakdown after NULL conversion:
data_quality_flag
NaN    136670
B       10061
Name: count, dtype: int64


## Summary and Descriptive Statistics

Let's verify the final cleaned dataset and compute descriptive statistics on `employment_thousands`.


In [ ]:
print(f"{'='*50}")
print(f"ILOSTAT Cleaned Dataset Summary")
print(f"{'='*50}")
print(f"Total rows:      {len(df):,}")
print(f"Countries:       {df['country_code'].nunique()}")
print(f"Years:           {df['year'].min()} – {df['year'].max()}")
print(f"ISIC sections:   {sorted(df['isic_section'].unique())}")
print(f"Sex categories:  {sorted(df['sex'].unique())}")
print(f"Break-in-series: {(df['data_quality_flag'] == 'B').sum():,} rows ({100*(df['data_quality_flag']=='B').mean():.1f}%)")
print()
print("Descriptive statistics for employment_thousands:")
print(df['employment_thousands'].describe().round(2))
print()
print("Sample rows:")
display(df.sample(5, random_state=42))

ILOSTAT Cleaned Dataset Summary
Total rows:      146,731
Countries:       209
Years:           1995 – 2025
ISIC sections:   ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U']
Sex categories:  ['F', 'M', 'T']
Break-in-series: 10,061 rows (6.9%)

Descriptive statistics for employment_thousands:
count    146731.00
mean        502.16
std        2929.05
min           0.00
25%          10.67
50%          58.70
75%         262.46
max      409881.80
Name: employment_thousands, dtype: float64

Sample rows:


,country_code,year,sex,isic_section,isic_version,employment_thousands,data_quality_flag
2835,ALB,2020,T,M,4,16.328,NaN
226142,NLD,2003,T,G,3,1245.755,NaN
34519,BIH,2017,M,I,4,21.688,NaN
197105,MDA,2008,T,O,3,35.678,NaN
107386,FRA,2022,M,C,4,2130.956,NaN


In [ ]:
df.head(10)

,country_code,year,sex,isic_section,isic_version,employment_thousands,data_quality_flag
331500,ZWE,2011,F,T,4,101.604,B
331499,ZWE,2011,F,S,4,78.642,B
331498,ZWE,2011,F,R,4,6.912,B
331497,ZWE,2011,F,Q,4,29.287,B
331496,ZWE,2011,F,P,4,92.091,B
331495,ZWE,2011,F,O,4,14.213,B
331494,ZWE,2011,F,N,4,21.721,B
331493,ZWE,2011,F,M,4,9.200,B
331491,ZWE,2011,F,K,4,7.770,B
331489,ZWE,2011,F,I,4,16.073,B


In [ ]:
# Export to CSV
OUTPUT = '/content/drive/MyDrive/CIS5500 Final Project Data/ilostat data/employment_cleaned.csv'
df.to_csv(OUTPUT, index=False)
print(f"Saved to: {OUTPUT}")

Saved to: /content/drive/MyDrive/CIS5500 Final Project Data/ilostat data/employment_cleaned.csv
